# Can we predict failure within 90 days of an inspection?
Target: `failed_90d`. Plan: inspect data, check for leakage, build baseline + gradient boosting, evaluate on a **time-based holdout** (train on earlier inspections, test on the latest).

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve
from sklearn.inspection import permutation_importance

df = pd.read_csv('data.csv', parse_dates=['inspected_at']).sort_values('inspected_at').reset_index(drop=True)
print(df.shape, 'assets:', df.asset_tag.nunique(), 'missing:', df.isna().sum().sum())
print('failure rate:', df.failed_90d.mean().round(4), '| range:', df.inspected_at.min(), '->', df.inspected_at.max())
df.describe().T

## Failure rate over time
The base rate is far from stable, so a single time split can be misleading.

In [ ]:
monthly = df.groupby(df.inspected_at.dt.to_period('M')).failed_90d.mean()
monthly.plot(kind='bar', figsize=(11,3), title='Failure rate by inspection month'); plt.ylabel('rate'); plt.show()

## Leakage check
`parts_replaced` is the number of parts replaced on the work order that *closed* the inspection record, so it is likely recorded after the outcome is known.

In [ ]:
df.groupby('parts_replaced').failed_90d.agg(['mean','size'])

Every record with 2+ parts replaced failed, and none with 0 did. That is near-deterministic, consistent with repairs following a failure, so it is **information not available at inspection time**. I exclude `parts_replaced`. `asset_tag` is unique per row (an identifier, no repeat history), so it is excluded too. Below I also show how misleading the leaky feature would be.

In [ ]:
feats_num = ['vibration_rms','bearing_temp_c','ambient_temp_c','meter_hours','load_pct']
feats_cat = ['site']
split = int(len(df)*0.75)
train, test = df.iloc[:split], df.iloc[split:]
print('train ends', train.inspected_at.max(), '| test starts', test.inspected_at.min(), '| test rate', test.failed_90d.mean().round(4))

def model(kind, nums, cats):
    pre = ColumnTransformer([('n', StandardScaler() if kind=='lr' else 'passthrough', nums),
                             ('c', OneHotEncoder(handle_unknown='ignore'), cats)])
    clf = (LogisticRegression(max_iter=1000) if kind=='lr' else
           HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=0))
    return make_pipeline(pre, clf)

def evaluate(name, m, nums, cats):
    m.fit(train[nums+cats], train.failed_90d)
    p = m.predict_proba(test[nums+cats])[:,1]
    return dict(model=name, roc_auc=roc_auc_score(test.failed_90d,p), pr_auc=average_precision_score(test.failed_90d,p)), p

rows, preds = [], {}
r, _ = evaluate('base rate (no skill)', make_pipeline(ColumnTransformer([('n','passthrough',feats_num)]), DummyClassifier()), feats_num, [])
r['pr_auc'] = test.failed_90d.mean()  # no-skill PR-AUC equals the prevalence
rows.append(r)
for name, kind, n, cat in [('logistic', 'lr', feats_num, feats_cat),
                           ('grad. boosting', 'gb', feats_num, feats_cat),
                           ('logistic + parts_replaced (LEAKY)', 'lr', feats_num+['parts_replaced'], feats_cat)]:
    r, p = evaluate(name, model(kind, n, cat), n, cat); rows.append(r); preds[name] = p
pd.DataFrame(rows).round(3)

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(11,4))
for k in ['logistic','grad. boosting']:
    f,t,_ = roc_curve(test.failed_90d, preds[k]); ax[0].plot(f,t,label=k)
    pr,rc,_ = precision_recall_curve(test.failed_90d, preds[k]); ax[1].plot(rc,pr,label=k)
ax[0].plot([0,1],[0,1],'k:'); ax[0].set(xlabel='FPR',ylabel='TPR',title='ROC (time holdout)')
ax[1].axhline(test.failed_90d.mean(),color='k',ls=':'); ax[1].set(xlabel='recall',ylabel='precision',title='Precision-recall')
ax[0].legend(); plt.tight_layout(); plt.show()

## Rolling-origin cross-validation
The last-25% test slice has a low base rate (~4%, few positives), so its AUC is noisy. Here each fold trains on all earlier data and tests on the next ~1/6 of the timeline.

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
res = []
for k, (tr, te) in enumerate(TimeSeriesSplit(n_splits=5).split(df)):
    for name, kind in [('logistic','lr'), ('grad. boosting','gb')]:
        mm = model(kind, feats_num, feats_cat).fit(df.iloc[tr][feats_num+feats_cat], df.failed_90d.iloc[tr])
        p = mm.predict_proba(df.iloc[te][feats_num+feats_cat])[:,1]
        y = df.failed_90d.iloc[te]
        res.append(dict(fold=k, model=name, test_start=df.inspected_at.iloc[te[0]].date(), base_rate=y.mean(),
                        roc_auc=roc_auc_score(y,p), pr_auc=average_precision_score(y,p)))
res = pd.DataFrame(res)
print(res.round(3).to_string(index=False))
res.groupby('model')[['roc_auc','pr_auc']].agg(['mean','std']).round(3)

## What drives risk?

In [ ]:
m = model('gb', feats_num, feats_cat).fit(train[feats_num+feats_cat], train.failed_90d)
pi = permutation_importance(m, test[feats_num+feats_cat], test.failed_90d, scoring='roc_auc', n_repeats=20, random_state=0)
imp = pd.Series(pi.importances_mean, index=feats_num+feats_cat).sort_values()
imp.plot.barh(title='Permutation importance (drop in test ROC-AUC)'); plt.show()
print(imp.round(4).to_string())
for f in ['vibration_rms','bearing_temp_c']:
    print(df.groupby(pd.qcut(df[f],5), observed=True).failed_90d.mean().round(3).to_string(), '\n')

## Practical value: triage the riskiest inspections

In [ ]:
t = test.assign(p=preds['grad. boosting']).sort_values('p', ascending=False)
for frac in [0.05,0.1,0.2,0.3]:
    top = t.head(int(len(t)*frac))
    print(f'top {frac:.0%}: failure rate {top.failed_90d.mean():.2f}  (lift {top.failed_90d.mean()/t.failed_90d.mean():.1f}x, captures {top.failed_90d.sum()/t.failed_90d.sum():.0%} of failures)')

## Caveats
- Labels near the end of the data window (last ~90 days) may be right-censored if later failures aren't recorded; the test slice includes these, which if anything understates performance.
- Failure rate swings from <1% to ~39% by month (clusters in summer 2024 and Jan 2025), so performance varies by period (fold ROC-AUC 0.65-0.84), and a model trained on little history may not transfer. Whether a time-varying driver (season, batch, site event) explains this is worth investigating.
- One inspection per asset, so an asset's own history (e.g. vibration trend) can't be used, likely a big missed opportunity.
- `parts_replaced` must not be used for prediction unless it is known at inspection time; confirm its timing with the data owner.